# DAC channel 1 â€” core bench tests
These cells contain the actual commands, measurements and assertions. They check:
1. SET / READ against the Keysight 34461A.
2. Ramp timing, sampled path and endpoints in both directions, with a 10Ã— speed change.
3. NETWORK preserves outputs and interrupts a ramp; SAFE ZERO works after closing Ethernet, at 0.01 V/s, then remains latched.

**All outputs must be isolated from equipment.** CH1 connects to the meter's voltage input and return. The other channels are checked digitally; this notebook cannot establish their analog accuracy with only CH1 connected. Test voltages stay within Â±0.05 V. The nominal DAC references remain Â±10 V even with Â±15 V power supplies.

USB actions need the optional `uno_r4_minima_bench` firmware. They enter the same action state machine as the buttons; they do not test contacts/debounce or electrically simulate a failed W5500. No serial-monitor application may hold the same COM port.

Run setup, then any test group. A failure gives a traceback and leaves the output for inspection. Each cell closes its connections even on failure; closing TCP interrupts a remote ramp when firmware detects the disconnect. It does **not** zero the output. A SAFE ZERO action already accepted continues independently. Successful tests explicitly finish CH1 at zero.


In [ ]:
from pathlib import Path
import sys, time, math, csv, json
from datetime import datetime, timezone
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "malab" / "instruments" / "ad5780.py").exists())
sys.path.insert(0, str(root))
import malab
assert Path(malab.__file__).resolve().parent == root / "malab", "Restart kernel: another malab is already imported"
from malab.instruments import AD5780, Keysight34461A, DACBenchControls
from malab.instruments.ad5780 import V_PER_BIT, voltage_to_code, code_to_voltage
from malab.instruments._line_socket import InstrumentError, InstrumentConnectionError, CommandError
import matplotlib.pyplot as plt
print("Driver checkout:", malab.__file__)


In [ ]:
# Change these only when the bench is ready. No hardware access occurs in setup.
RUN_LIVE = False
ALL_OUTPUTS_ISOLATED = False
DAC_ADDRESS = "192.168.1.200"
METER_ADDRESS = "192.168.1.71"
USB_PORT = "COM5"  # Check on the computer running this notebook.
CHANNEL = 1
MAX_TEST_V = 0.05
VOLTAGE_TOL_V = 0.0005  # 0.5 mV bench acceptance, not a calibration specification.
RAMP_SLOW_TOL = 0.25   # At most 25% slow, plus sampling/communication allowance.
SAMPLE_PAUSE_S = 0.02
RESULTS = root / "malab" / "notebooks" / "results"
traces = {}
print("Live tests enabled:", RUN_LIVE, "| isolated:", ALL_OUTPUTS_ISOLATED)


In [ ]:
def ready():
    assert RUN_LIVE and ALL_OUTPUTS_ISOLATED, "Bench paused: confirm setup, then enable both flags"
    assert CHANNEL == 1, "This wiring/test plan is for channel 1"
    assert 0 < MAX_TEST_V <= 0.05

def connect_dac_after_reset(timeout=20):
    # Retry connection/ID only. Never replay SET, RAMP, INIT, or USB actions.
    deadline = time.monotonic() + timeout
    last = None
    while time.monotonic() < deadline:
        dac = None
        try:
            dac = AD5780(address=DAC_ADDRESS, timeout=min(1, deadline-time.monotonic()))
            assert dac.get_id() == "AD5780"
            dac.set_timeout(3)
            return dac
        except (InstrumentError, OSError) as exc:
            last = exc
            if dac is not None:
                dac.close()
            time.sleep(0.2)
    raise TimeoutError(f"No fresh DAC connection within {timeout}s: {last}")

def save_trace(name, rows):
    # Logging only. All test commands/assertions remain in the test cells below.
    traces[name] = list(rows)
    RESULTS.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    path = RESULTS / f"{stamp}_{name}.csv"
    if rows:
        with path.open("w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
    print("Trace:", path, "| samples:", len(rows))


## Connection and meter check â€” no DAC output writes
The meter is put into DC-voltage measurement mode. This cell performs ID and register reads only.


In [ ]:
ready()
with AD5780(address=DAC_ADDRESS) as dac, Keysight34461A(address=METER_ADDRESS) as meter:
    print("DAC:", dac.get_id())
    print("Meter:", meter.get_id())
    meter.configure_dc_voltage(10, nplc=1)
    code_now = dac.get_code(CHANNEL)
    measured = meter.read_voltage()
    print("CH1 code:", code_now, "| nominal V:", code_to_voltage(code_now), "| measured V:", measured)
    meter.check_errors()


## 1. SET / GET / multimeter agreement
Test both polarities and zero. The exact register code is checked separately from the analog tolerance. INIT explicitly verifies/configures CH1; it preserves an already active output. If initial readback or meter voltage lies outside the small test window, this cell stops before SET.


In [ ]:
ready()
rows = []
try:
    with AD5780(address=DAC_ADDRESS) as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10, nplc=1)
        assert abs(dac.get_voltage(CHANNEL)) <= MAX_TEST_V + V_PER_BIT
        assert abs(meter.read_voltage()) <= MAX_TEST_V + VOLTAGE_TOL_V
        print(dac.initialize(CHANNEL))
        for requested in (0, 0.01, 0.05, 0, -0.01, -0.05, 0):
            assert abs(requested) <= MAX_TEST_V
            quantized = dac.set_voltage(CHANNEL, requested)
            time.sleep(0.15)
            actual_code = dac.get_code(CHANNEL)
            reported = dac.get_voltage(CHANNEL)
            readings = [meter.read_voltage() for _ in range(3)]
            measured = sum(readings) / len(readings)
            row = dict(requested_V=requested, code=actual_code, reported_V=reported,
                       measured_V=measured, error_V=measured-quantized)
            rows.append(row)
            print(row)
            assert actual_code == voltage_to_code(requested), row
            assert reported == quantized, row
            assert all(abs(v-quantized) <= VOLTAGE_TOL_V for v in readings), row
        meter.check_errors()
finally:
    save_trace("set_get_voltage", rows)
print("PASS: SET, GET and CH1 analog readings agree within", VOLTAGE_TOL_V, "V")


## 2. Ramp timing, path and endpoints â€” 0.001 versus 0.01 V/s
Each run covers 0.02 V: about 20 s versus 2 s. Both ascending and descending paths are measured. Delay is computed in microseconds and checked against the quantized number of steps.

Meter readings are timestamped around the acquisition; midpoint and acquisition width describe timing uncertainty. The path allowance includes one DAC step, meter tolerance, acquisition width, and the configured 25% slowness allowance. This detects the former factor-of-10/1-second issue; it is not a measurement of every individual 76 ÂµV DAC transition. The status polling and SPI service overhead may make firmware slower than nominal; the tests report that as a failure rather than quietly widening limits.


In [ ]:
ready()
ramp_rows, ramp_summaries = [], []
try:
    with AD5780(address=DAC_ADDRESS) as dac, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10, nplc=0.2)
        assert abs(dac.get_voltage(CHANNEL)) <= MAX_TEST_V + V_PER_BIT
        assert abs(meter.read_voltage()) <= MAX_TEST_V + VOLTAGE_TOL_V
        dac.initialize(CHANNEL)
        for direction, start_v, target_v in [("up", 0.0, 0.02), ("down", 0.0, -0.02)]:
            for speed in (0.001, 0.01):
                assert max(abs(start_v), abs(target_v)) <= MAX_TEST_V
                v0 = dac.set_voltage(CHANNEL, start_v)
                time.sleep(0.15)
                start_code, target_code = dac.get_code(CHANNEL), voltage_to_code(target_v)
                end_v = code_to_voltage(target_code)
                assert abs(meter.read_voltage()-v0) <= VOLTAGE_TOL_V
                step = 10
                delay_us = math.ceil(step * V_PER_BIT / speed * 1e6)
                expected_s = math.ceil(abs(target_code-start_code)/step) * delay_us / 1e6
                run = f"{direction}_{speed:g}"
                points = []
                t0 = time.monotonic()
                ack = dac.ramp(CHANNEL, target_v, speed, step_size=step, wait=False)
                assert ack.state == "ACCEPTED"
                while True:
                    ta = time.monotonic()
                    measured = meter.read_voltage()
                    tb = time.monotonic()
                    status = dac.get_status()
                    row = dict(run=run, elapsed_s=(ta+tb)/2-t0, aperture_s=tb-ta,
                               measured_V=measured, state=status.state, speed_V_s=speed,
                               start_V=v0, target_V=end_v)
                    points.append(row)
                    ramp_rows.append(row)
                    assert status.channel == CHANNEL and status.target_code == target_code, status
                    assert status.state in ("RUNNING", "COMPLETE"), status
                    if status.state == "COMPLETE":
                        elapsed = time.monotonic()-t0
                        break
                    assert time.monotonic()-t0 < expected_s*2+5, "Ramp exceeded 2Ã— expected duration"
                    time.sleep(SAMPLE_PAUSE_S)
                final_code = dac.get_code(CHANNEL)
                final_v = meter.read_voltage()
                gap = max([p["aperture_s"] for p in points] +
                          [b["elapsed_s"]-a["elapsed_s"] for a,b in zip(points, points[1:])])
                summary = dict(run=run, expected_s=expected_s, elapsed_s=elapsed,
                               ratio=elapsed/expected_s, final_code=final_code, measured_V=final_v)
                ramp_summaries.append(summary)
                print(summary)
                assert final_code == target_code, summary
                assert abs(final_v-end_v) <= VOLTAGE_TOL_V, summary
                assert expected_s-0.1 <= elapsed <= expected_s*(1+RAMP_SLOW_TOL)+2*gap+0.25, summary
                assert len(points) >= 5, "Insufficient temporal sampling"
                sign = 1 if end_v > v0 else -1
                for p in points:
                    ideal = v0 + sign * min(abs(end_v-v0), speed*p["elapsed_s"])
                    path_tol = VOLTAGE_TOL_V + step*V_PER_BIT + speed*p["aperture_s"] + RAMP_SLOW_TOL*abs(end_v-v0)
                    assert abs(p["measured_V"]-ideal) <= path_tol, (p, ideal, path_tol)
                    assert min(v0,end_v)-VOLTAGE_TOL_V <= p["measured_V"] <= max(v0,end_v)+VOLTAGE_TOL_V, p
                for a,b in zip(points, points[1:]):
                    assert sign*(b["measured_V"]-a["measured_V"]) >= -2*VOLTAGE_TOL_V, (a,b)
                dac.set_voltage(CHANNEL, 0)
        for direction in ("up", "down"):
            slow = next(r for r in ramp_summaries if r["run"] == direction+"_0.001")
            fast = next(r for r in ramp_summaries if r["run"] == direction+"_0.01")
            assert 7 <= slow["elapsed_s"]/fast["elapsed_s"] <= 13, (slow,fast)
        meter.check_errors()
finally:
    save_trace("ramp_paths", ramp_rows)
    save_trace("ramp_timing", ramp_summaries)
print("PASS: both directions, endpoints, sampled path and 10Ã— speed change")


In [ ]:
# Run even after a failed ramp assertion to inspect the captured trace.
for run in sorted({p["run"] for p in traces.get("ramp_paths", [])}):
    points = [p for p in traces["ramp_paths"] if p["run"] == run]
    plt.plot([p["elapsed_s"] for p in points], [p["measured_V"] for p in points], ".-", label=run)
plt.xlabel("Seconds from RAMP request"); plt.ylabel("Measured CH1 voltage (V)")
plt.grid(); plt.legend(); plt.show()


## 3a. NETWORK reset â€” preserve all four DAC codes and sampled CH1 voltage
Reset once through USB, require the old connection to fail, then establish a fresh connection. The short measurement series checks for lasting changes; it cannot exclude transients between meter samples.


In [ ]:
ready()
rows = []
try:
    with DACBenchControls(USB_PORT) as controls, AD5780(address=DAC_ADDRESS) as old, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10, nplc=0.2)
        assert abs(old.get_voltage(CHANNEL)) <= MAX_TEST_V + V_PER_BIT
        assert abs(meter.read_voltage()) <= MAX_TEST_V + VOLTAGE_TOL_V
        old.initialize(CHANNEL)
        held = old.set_voltage(CHANNEL, 0.03)
        time.sleep(0.15)
        before = [old.get_code(ch) for ch in range(1,5)]
        baseline = meter.read_voltage()
        assert abs(baseline-held) <= VOLTAGE_TOL_V
        t0 = time.monotonic()
        print(controls.network_reset())
        for _ in range(12):
            rows.append(dict(elapsed_s=time.monotonic()-t0, measured_V=meter.read_voltage()))
            time.sleep(0.05)
        stale_failed = False
        try:
            old.get_id()
        except (InstrumentError, OSError):
            stale_failed = True
        assert stale_failed, "Old TCP connection unexpectedly survived NETWORK"
        old.close()
        with connect_dac_after_reset() as fresh:
            after = [fresh.get_code(ch) for ch in range(1,5)]
            assert after == before, (before, after)
            assert all(abs(p["measured_V"]-baseline) <= 2*VOLTAGE_TOL_V for p in rows), rows
            assert abs(meter.read_voltage()-held) <= VOLTAGE_TOL_V
            fresh.set_voltage(CHANNEL, 0)
finally:
    save_trace("network_preserves_output", rows)
print("PASS: old session invalidated, fresh session works, DAC codes preserved")


## 3b. NETWORK during a remote ramp â€” abort and hold
Wait for CH1 to move measurably toward +0.05 V before resetting. The output must then stop at an intermediate value, stay there, and agree with the new connection's readback.


In [ ]:
ready()
rows = []
try:
    with DACBenchControls(USB_PORT) as controls, AD5780(address=DAC_ADDRESS) as old, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10, nplc=0.2)
        assert abs(old.get_voltage(CHANNEL)) <= MAX_TEST_V + V_PER_BIT
        assert abs(meter.read_voltage()) <= MAX_TEST_V + VOLTAGE_TOL_V
        old.initialize(CHANNEL)
        old.set_voltage(CHANNEL, 0)
        other_before = [old.get_code(ch) for ch in (2,3,4)]
        old.ramp(CHANNEL, 0.05, 0.01, wait=False)
        deadline = time.monotonic()+6
        while True:
            v = meter.read_voltage()
            if v >= 0.01:
                break
            assert time.monotonic() < deadline, "Ramp never reached +10 mV"
            time.sleep(0.03)
        print(controls.network_reset())
        old.close()
        time.sleep(0.2)  # Cancellation/transport latency excluded from hold window.
        for _ in range(12):
            rows.append(dict(timestamp_s=time.monotonic(), measured_V=meter.read_voltage()))
            time.sleep(0.05)
        with connect_dac_after_reset() as fresh:
            status = fresh.get_status()
            assert status.state == "ABORTED", status
            held = fresh.get_voltage(CHANNEL)
            assert 0.005 < held < 0.045, f"Expected an intermediate voltage; got {held}"
            assert all(abs(p["measured_V"]-held) <= VOLTAGE_TOL_V for p in rows), rows
            assert [fresh.get_code(ch) for ch in (2,3,4)] == other_before
            fresh.set_voltage(CHANNEL, 0)
finally:
    save_trace("network_interrupts_ramp", rows)
print("PASS: NETWORK aborts the ramp and holds its intermediate output")


## 3c. SAFE ZERO without a DAC Ethernet session â€” both polarities
CH1 starts at +0.05 V and then -0.05 V. Other channels are not deliberately raised; all four must start within the small test window. Close the TCP connection, invoke SAFE ZERO over USB, measure the ramp, verify a NETWORK request is rejected during the ramp, and verify the latch persists.

Only a fresh NETWORK action may rearm. After rearming, all four codes must be exactly 131072. Invalid readback can also produce a latched state; checking timing, CH1 voltage and all four final codes prevents mistaking that state for successful zeroing.


In [ ]:
ready()
rows, summaries = [], []
try:
    with DACBenchControls(USB_PORT) as controls, Keysight34461A(address=METER_ADDRESS) as meter:
        meter.configure_dc_voltage(10, nplc=0.2)
        for start_v in (0.05, -0.05):
            with connect_dac_after_reset() as dac:
                # Configure only cold/clamped channels at zero; active channels are preserved.
                for ch in range(1,5):
                    dac.initialize(ch)
                before = [dac.get_code(ch) for ch in range(1,5)]
                assert all(abs(code_to_voltage(c)) <= MAX_TEST_V+V_PER_BIT for c in before), before
                assert abs(meter.read_voltage()) <= MAX_TEST_V+VOLTAGE_TOL_V
                dac.initialize(CHANNEL)
                v0 = dac.set_voltage(CHANNEL, start_v)
                time.sleep(0.15)
                assert abs(meter.read_voltage()-v0) <= VOLTAGE_TOL_V
            # TCP is closed now; SAFE ZERO is independent of the client connection.
            t0 = time.monotonic()
            print(controls.safe_zero())
            rejected = False
            try:
                controls.network_reset()
            except CommandError:
                rejected = True
            assert rejected, "NETWORK must not cancel/rearm an active SAFE ZERO"
            points = []
            expected_s = abs(v0)/0.01
            while True:
                ta = time.monotonic()
                measured = meter.read_voltage()
                tb = time.monotonic()
                state = controls.status()
                row = dict(start_V=v0, elapsed_s=(ta+tb)/2-t0, aperture_s=tb-ta,
                           measured_V=measured, state=state)
                points.append(row); rows.append(row)
                assert state in ("ACTIVE", "LATCHED"), state
                if state == "LATCHED":
                    elapsed = time.monotonic()-t0
                    break
                assert time.monotonic()-t0 < expected_s*2+5, "SAFE ZERO too slow"
                time.sleep(SAMPLE_PAUSE_S)
            summary = dict(start_V=v0, expected_s=expected_s, elapsed_s=elapsed, final_V=meter.read_voltage())
            summaries.append(summary)
            print(summary)
            gap = max([p["aperture_s"] for p in points] +
                      [b["elapsed_s"]-a["elapsed_s"] for a,b in zip(points,points[1:])])
            assert expected_s-0.1 <= elapsed <= expected_s*(1+RAMP_SLOW_TOL)+2*gap+0.25, summary
            assert abs(summary["final_V"]) <= VOLTAGE_TOL_V
            assert len(points) >= 5
            sign = 1 if v0 > 0 else -1
            for p in points:
                ideal = sign*max(0,abs(v0)-0.01*p["elapsed_s"])
                tolerance = VOLTAGE_TOL_V+V_PER_BIT+0.01*p["aperture_s"]+RAMP_SLOW_TOL*abs(v0)
                assert abs(p["measured_V"]-ideal) <= tolerance, (p,ideal,tolerance)
                assert min(v0,0)-VOLTAGE_TOL_V <= p["measured_V"] <= max(v0,0)+VOLTAGE_TOL_V, p
            for a,b in zip(points,points[1:]):
                assert sign*(b["measured_V"]-a["measured_V"]) <= 2*VOLTAGE_TOL_V, (a,b)
            time.sleep(0.5)
            assert controls.status() == "LATCHED", "SAFE ZERO rearmed without a NETWORK action"
            # A latched controller must not accept a normal command session.
            connected_while_latched = False
            try:
                with AD5780(address=DAC_ADDRESS, timeout=0.5) as blocked:
                    connected_while_latched = blocked.get_id() == "AD5780"
            except (InstrumentError, OSError):
                pass
            assert not connected_while_latched
            print(controls.network_reset())  # Deliberate rearm after completion.
            with connect_dac_after_reset() as dac:
                final_codes = [dac.get_code(ch) for ch in range(1,5)]
                assert final_codes == [131072]*4, final_codes
                assert abs(meter.read_voltage()) <= VOLTAGE_TOL_V
        meter.check_errors()
finally:
    save_trace("safe_zero_paths", rows)
    save_trace("safe_zero_timing", summaries)
print("PASS: SAFE ZERO slopes/endpoints, network independence, priority, latch, and rearm")


In [ ]:
for start in sorted({p["start_V"] for p in traces.get("safe_zero_paths", [])}):
    points = [p for p in traces["safe_zero_paths"] if p["start_V"] == start]
    plt.plot([p["elapsed_s"] for p in points], [p["measured_V"] for p in points], ".-", label=f"start {start:g} V")
plt.xlabel("Seconds from SAFE ZERO request"); plt.ylabel("Measured CH1 voltage (V)")
plt.grid(); plt.legend(); plt.show()


## Manual button check after automated action tests
With the same isolated setup, repeat NETWORK and SAFE ZERO using **A1â†’GND** and **A2â†’GND** respectively. USB emulation validates the shared action code but not physical wiring, debounce, or a held contact.

No measurements in a saved notebook imply a live pass unless that test cell has actually run. CSV traces in `notebooks/results/` are local records and are excluded from Git.
